In [29]:
import numpy as np
import pandas as pd

np.random.seed(0)

df = pd.DataFrame({
    "score": np.random.normal(600, 50, 10000)
})

feature = "score"

# 1. round
df[feature] = df[feature].round(6)

# 2. bins
cats, bins = pd.qcut(
    df[feature],
    q=10,
    duplicates="drop",
    retbins=True
)

n_bins = len(bins) - 1

# 3. counts (POSITIONAL, no index alignment)
counts = cats.value_counts().sort_index().values

# 4. build df
stability_df = pd.DataFrame({
    "quantile_start": bins[:-1].round(6),
    "quantile_end": bins[1:].round(6),
    "expected_count": counts,
    "expected_percent": counts / len(df)
})

# 5. correct pandas qcut flags
stability_df["SEXCL"] = 1
stability_df["EEXCL"] = 0
stability_df.loc[0, "SEXCL"] = 0
stability_df['variable_name'] = 'score'
stability_df['idx'] = stability_df.index + 1

stability_df = stability_df[
    ["idx","variable_name","quantile_start","quantile_end","SEXCL","EEXCL",
     "expected_count","expected_percent"]
]

stability_df


,idx,variable_name,quantile_start,quantile_end,SEXCL,EEXCL,expected_count,expected_percent
0,1,score,412.994968,534.988319,0,0,1000,0.1
1,2,score,534.988319,557.186645,1,0,1000,0.1
2,3,score,557.186645,573.412609,1,0,1000,0.1
3,4,score,573.412609,586.805514,1,0,1000,0.1
4,5,score,586.805514,598.646370,1,0,1000,0.1
5,6,score,598.646370,610.923908,1,0,1000,0.1
6,7,score,610.923908,624.551238,1,0,1000,0.1
7,8,score,624.551238,640.549153,1,0,1000,0.1
8,9,score,640.549153,662.731725,1,0,1000,0.1
9,10,score,662.731725,790.083011,1,0,1000,0.1


In [30]:
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor


def stability_index_pandas(
    df: pd.DataFrame,
    features: list[str],
    max_bins: int = 10,
    round_digits: int = 6,
    n_jobs: int = 4
) -> pd.DataFrame:
    """
    Build stability index template for multiple continuous features using pandas qcut.

    Parameters
    ----------
    df : pd.DataFrame
    features : list[str]
        Continuous columns to bin
    max_bins : int
        Maximum quantile bins
    round_digits : int
        Decimal rounding precision
    n_jobs : int
        Thread pool workers

    Returns
    -------
    pd.DataFrame
        Concatenated stability index table for all features
    """

    def process_feature(feature: str) -> pd.DataFrame:
        s = df[feature].dropna().round(round_digits)

        # ---- qcut ----
        cats, bins = pd.qcut(
            s,
            q=max_bins,
            duplicates="drop",
            retbins=True
        )

        counts = cats.value_counts().sort_index().to_numpy()

        n_bins = len(bins) - 1

        out = pd.DataFrame({
            "idx": np.arange(1, n_bins + 1),
            "variable_name": feature,
            "quantile_start": bins[:-1].round(round_digits),
            "quantile_end": bins[1:].round(round_digits),
            "expected_count": counts,
            "expected_percent": counts / len(s)
        })

        # ---- pandas qcut semantics ----
        # first bin [start, end], others (start, end]
        out["SEXCL"] = 1
        out["EEXCL"] = 0
        out.loc[0, "SEXCL"] = 0

        return out[
            ["idx", "variable_name", "quantile_start", "quantile_end",
             "SEXCL", "EEXCL", "expected_count", "expected_percent"]
        ]

    # ---- parallel execution ----
    with ThreadPoolExecutor(max_workers=n_jobs) as executor:
        results = list(executor.map(process_feature, features))

    # ---- concatenate all ----
    final_df = pd.concat(results, ignore_index=True)

    return final_df


In [41]:
import numpy as np
import pandas as pd
from concurrent.futures import ThreadPoolExecutor


# =========================================================
# PSI TABLE BUILDER (THREADED + CORRECTED BOUNDARIES)
# =========================================================
def psi_table_pandas(
    actual_df: pd.DataFrame,
    stability_df: pd.DataFrame,
    features: list[str],
    round_digits: int = 6,
    n_jobs: int = 4
) -> pd.DataFrame:
    """
    Build PSI table using stability index bins.

    For each feature:

    1. Reuse quantile_start / quantile_end from stability table
    2. Extend bins with -inf and +inf to catch unseen values
    3. Count actual observations using pd.cut
    4. Fix boundary mismatch carefully:
         ONLY values exactly equal to first boundary are shifted
         NOT all overflow values
    5. Join expected + actual side-by-side
    6. Run features in parallel with ThreadPoolExecutor

    Final bins:

        (-inf, s1)          <- overflow-left
        [s1, e1]
        (e1, e2]
        ...
        (e10, +inf]         <- overflow-right

    Note:
        Because we manually move boundary values, first bin is
        end EXCLUDED → hence EEXCL = 1
    """

    # -----------------------------------------------------
    # worker for single feature
    # -----------------------------------------------------
    def process_feature(feature):

        # ---- extract stability rows for this feature ----
        stab = (
            stability_df[stability_df["variable_name"] == feature]
            .sort_values("idx")
            .reset_index(drop=True)
        )

        starts = stab["quantile_start"].to_numpy()
        ends   = stab["quantile_end"].to_numpy()

        # -------------------------------------------------
        # Create extended bins
        # [-inf, s1, s2, ..., last_end, +inf]
        # -------------------------------------------------
        bins = np.concatenate((
            [-np.inf],
            starts,
            [ends[-1]],
            [np.inf]
        ))

        bins = np.unique(bins)

        # -------------------------------------------------
        # CUT actual data
        # -------------------------------------------------
        actual = actual_df[feature].round(round_digits)

        cats = pd.cut(
            actual,
            bins=bins,
            right=True,          # (a, b]
            include_lowest=True
        )

        counts = cats.value_counts().sort_index().to_numpy()

        # -------------------------------------------------
        # CRITICAL FIX (boundary leakage)
        #
        # qcut first bin originally:  [s1, e1]
        # new bins create:           (-inf, s1]
        #
        # values == s1 wrongly move to overflow.
        #
        # We ONLY move those exact matches back.
        # -------------------------------------------------
        s1 = starts[0]

        leak_count = (actual == s1).sum()

        counts[0] -= leak_count      # remove from overflow
        counts[1] += leak_count      # add to first real bin


        # -------------------------------------------------
        # build actual part
        # -------------------------------------------------
        psi_part = pd.DataFrame({
            "idx": np.arange(0, len(counts)),
            "variable_name": feature,
            "actual_count": counts,
            "actual_percent": counts / len(actual)
        })


        # -------------------------------------------------
        # extend stability table with 2 overflow rows
        #
        # IMPORTANT FLAG FIX:
        #
        # first bin = (-inf, s1)
        #   start included, end EXCLUDED
        #   SEXCL=0, EEXCL=1  <-- corrected
        #
        # last bin = (last_end, +inf]
        # -------------------------------------------------
        overflow_rows = pd.DataFrame({
            "idx": [0, len(stab) + 1],
            "variable_name": feature,
            "quantile_start": [-np.inf, ends[-1]],
            "quantile_end": [starts[0], np.inf],
            "SEXCL": [0, 1],
            "EEXCL": [1, 0],   # ✅ FIXED (first bin end excluded)
            "expected_count": [0, 0],
            "expected_percent": [0.0, 0.0]
        })

        stab_ext = pd.concat(
            [overflow_rows.iloc[[0]], stab, overflow_rows.iloc[[1]]],
            ignore_index=True
        )

        stab_ext["idx"] = np.arange(0, len(stab_ext))

        # -------------------------------------------------
        # join expected + actual
        # -------------------------------------------------
        final = stab_ext.merge(
            psi_part,
            on=["idx", "variable_name"],
            how="left"
        )

        return final


    # -----------------------------------------------------
    # parallel execution
    # -----------------------------------------------------
    with ThreadPoolExecutor(max_workers=n_jobs) as executor:
        results = list(executor.map(process_feature, features))

    return pd.concat(results, ignore_index=True)

In [42]:
psi_df = psi_table_pandas(
    actual_df=df,
    stability_df=stability_df,
    features=["feature1", "feature2", "feature3"],
    n_jobs=3
)

psi_df


,idx,variable_name,quantile_start,quantile_end,SEXCL,EEXCL,expected_count,expected_percent,actual_count,actual_percent
0,0,feature1,-inf,403.879987,0,1,0,0.0,0,0.0
1,1,feature1,403.879987,535.385052,0,0,1000,0.1,1000,0.1
2,2,feature1,535.385052,557.777624,1,0,1000,0.1,1000,0.1
3,3,feature1,557.777624,573.564670,1,0,1000,0.1,1000,0.1
4,4,feature1,573.564670,587.365989,1,0,1000,0.1,1000,0.1
5,5,feature1,587.365989,599.870251,1,0,1000,0.1,1000,0.1
6,6,feature1,599.870251,612.492288,1,0,1000,0.1,1000,0.1
7,7,feature1,612.492288,625.751949,1,0,1000,0.1,1000,0.1
8,8,feature1,625.751949,641.968650,1,0,1000,0.1,1000,0.1
9,9,feature1,641.968650,664.063959,1,0,1000,0.1,1000,0.1


In [43]:
# =========================================================
# TEST DATA GENERATOR (drift scenarios)
# =========================================================
def generate_test_datasets(n=10000, seed=42):
    np.random.seed(seed)

    base = pd.DataFrame({
        "feature1": np.random.normal(600, 50, n),
        "feature2": np.random.lognormal(5.5, 0.5, n),
        "feature3": 1000 * np.random.beta(5, 2, n)
    })

    no_shift = base.copy()

    left_shift = pd.DataFrame({
        "feature1": base["feature1"] - 40,
        "feature2": base["feature2"] * 0.7,
        "feature3": base["feature3"] * 0.85
    })

    right_shift = pd.DataFrame({
        "feature1": base["feature1"] + 40,
        "feature2": base["feature2"] * 1.4,
        "feature3": base["feature3"] * 1.15
    })

    return base, no_shift, left_shift, right_shift

In [44]:
base, no_shift, left_shift, right_shift = generate_test_datasets()

features = ["feature1", "feature2", "feature3"]

stab_df = stability_index_pandas(base, features)

psi_no   = psi_table_pandas(no_shift, stab_df, features)
psi_left = psi_table_pandas(left_shift, stab_df, features)
psi_right= psi_table_pandas(right_shift, stab_df, features)


In [49]:
stab_df

,idx,variable_name,quantile_start,quantile_end,SEXCL,EEXCL,expected_count,expected_percent
0,1,feature1,403.879987,535.385052,0,0,1000,0.1
1,2,feature1,535.385052,557.777624,1,0,1000,0.1
2,3,feature1,557.777624,573.564670,1,0,1000,0.1
3,4,feature1,573.564670,587.365989,1,0,1000,0.1
4,5,feature1,587.365989,599.870251,1,0,1000,0.1
5,6,feature1,599.870251,612.492288,1,0,1000,0.1
6,7,feature1,612.492288,625.751949,1,0,1000,0.1
7,8,feature1,625.751949,641.968650,1,0,1000,0.1
8,9,feature1,641.968650,664.063959,1,0,1000,0.1
9,10,feature1,664.063959,796.311885,1,0,1000,0.1


In [57]:
# get first boundary (s1)
s1 = (
    stab_df[stab_df["variable_name"] == "feature1"]
    .sort_values("idx")
    ["quantile_start"]
    .iloc[0]
)

print("Boundary s1 =", s1)


Boundary s1 = 403.879987


In [58]:
left_shift_test = left_shift.copy()

n_inject = 25   # number of boundary test rows

left_shift_test.loc[:n_inject-1, "feature1"] = s1


In [59]:
(left_shift_test["feature1"] == s1).sum()

np.int64(25)

In [64]:
psi_left = psi_table_pandas(left_shift_test, stab_df, features)

In [66]:
# left_shift['feature1'].min()

In [70]:
left_shift_test[(left_shift_test['feature1']>403.879987) & (left_shift_test['feature1']<=535.385052)].shape[0]

3102

In [72]:
left_shift_test[(left_shift_test['feature1']>=-np.inf) & (left_shift_test['feature1']<403.879987)].shape[0]

10

In [63]:
left_shift_test[left_shift_test['feature1'] == 403.879987].shape[0]

25

,idx,variable_name,quantile_start,quantile_end,SEXCL,EEXCL,expected_count,expected_percent,actual_count,actual_percent
0,0,feature1,-inf,403.879987,0,1,0,0.0,10,0.0010
1,1,feature1,403.879987,535.385052,0,0,1000,0.1,3110,0.3110
2,2,feature1,535.385052,557.777624,1,0,1000,0.1,1706,0.1706
3,3,feature1,557.777624,573.564670,1,0,1000,0.1,1252,0.1252
4,4,feature1,573.564670,587.365989,1,0,1000,0.1,1025,0.1025
5,5,feature1,587.365989,599.870251,1,0,1000,0.1,783,0.0783
6,6,feature1,599.870251,612.492288,1,0,1000,0.1,632,0.0632
7,7,feature1,612.492288,625.751949,1,0,1000,0.1,532,0.0532
8,8,feature1,625.751949,641.968650,1,0,1000,0.1,445,0.0445
9,9,feature1,641.968650,664.063959,1,0,1000,0.1,319,0.0319
